In [1]:
import numpy as np
import pandas as pd

**1**

In [4]:
data = [
    {"місто": "Одеса", "ціна": 640, "кількість": 5},
    {"місто": " Одеса ", "ціна": "640 грн", "кількість": np.nan},
    {"місто": "ОДЕСА", "ціна": 640, "кількість": 5},
    {"місто": "ОДЕСА", "ціна": 640, "кількість": 5},
    {"місто": "Одеса", "ціна": "640 грн", "кількість": 4},
    {"місто": "Одеса", "ціна": 6400, "кількість": 5},
    {"місто": "Одеса", "ціна": 640, "кількість": np.nan},
    {"місто": "Одеса", "ціна": 640, "кількість": 6},
]

df = pd.DataFrame(data)
print("--- вхідний брудний датафрейм ---")
print(df)
print("\n" + "+"*50 + "\n")

--- вхідний брудний датафрейм ---
     місто     ціна  кількість
0    Одеса      640        5.0
1   Одеса   640 грн        NaN
2    ОДЕСА      640        5.0
3    ОДЕСА      640        5.0
4    Одеса  640 грн        4.0
5    Одеса     6400        5.0
6    Одеса      640        NaN
7    Одеса      640        6.0

++++++++++++++++++++++++++++++++++++++++++++++++++



**2**

In [5]:
print("--- 1. Кількість пропусків у стовпцях ---")
print(df.isna().sum())

median_qty = df["кількість"].median()
df["кількість"].fillna(median_qty, inplace=True)

print(f"\nМедіана для стовпця 'кількість': {median_qty}")
print("Датафрейм після заповнення пропусків:")
print(df)
print("\n" + "="*50 + "\n")

--- 1. Кількість пропусків у стовпцях ---
місто        0
ціна         0
кількість    2
dtype: int64

Медіана для стовпця 'кількість': 5.0
Датафрейм після заповнення пропусків:
     місто     ціна  кількість
0    Одеса      640        5.0
1   Одеса   640 грн        NaN
2    ОДЕСА      640        5.0
3    ОДЕСА      640        5.0
4    Одеса  640 грн        4.0
5    Одеса     6400        5.0
6    Одеса      640        NaN
7    Одеса      640        6.0




C:\Users\Користувач\AppData\Local\Temp\ipykernel_15380\375329440.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["кількість"].fillna(median_qty, inplace=True)


**3**

In [7]:
print("Дублікати без subset (повний збіг):", df.duplicated().sum())
print("Дублікати з subset=['місто', 'ціна']:", df.duplicated(subset=["місто", "ціна"]).sum())

df.drop_duplicates(inplace=True)
print(f"\nКількість рядків після видалення дублікатів: {len(df)}")
print("\n" + "="*50 + "\n")

Дублікати без subset (повний збіг): 1
Дублікати з subset=['місто', 'ціна']: 3

Кількість рядків після видалення дублікатів: 7




**4**

In [8]:
df["ціна"] = df["ціна"].astype(str).str.replace("грн", "").str.strip().astype(float)

df["місто"] = df["місто"].str.strip().str.title()

print("Унікальні значення міста після стандартизації:", df["місто"].unique())
print("типи даних після очищення:")
print(df.dtypes)
print("\nОчищений датафрейм:")
print(df)
print("\n" + "="*50 + "\n")

Унікальні значення міста після стандартизації: <StringArray>
['Одеса']
Length: 1, dtype: str
типи даних після очищення:
місто            str
ціна         float64
кількість    float64
dtype: object

Очищений датафрейм:
   місто    ціна  кількість
0  Одеса   640.0        5.0
1  Одеса   640.0        NaN
2  Одеса   640.0        5.0
4  Одеса   640.0        4.0
5  Одеса  6400.0        5.0
6  Одеса   640.0        NaN
7  Одеса   640.0        6.0




**5**

In [9]:
q1 = df["ціна"].quantile(0.25)
q3 = df["ціна"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1: {q1}, Q3: {q3}, IQR: {iqr}")
print(f"Нижня межа: {lower_bound}, Верхня межа: {upper_bound}")

outliers = df[(df["ціна"] < lower_bound) | (df["ціна"] > upper_bound)]
print("\nВиявлені викиди за методом IQR:")
print(outliers)

Q1: 640.0, Q3: 640.0, IQR: 0.0
Нижня межа: 640.0, Верхня межа: 640.0

Виявлені викиди за методом IQR:
   місто    ціна  кількість
5  Одеса  6400.0        5.0


### Відповіді на підсумкові питання для контролю

1. **Чому заповнення середнім може спотворити стандартне відхилення?**
* Заповнення пропусків середнім штучно зменшує розсіювання даних навколо центру (варіативність), оскільки додає значення з нульовим відхиленням від середнього. У результаті обчислена дисперсія та стандартне відхилення будуть **заниженими** порівняно з реальними.


2. **Яка різниця між `duplicated()` без `subset` і з `subset`?**
* Без `subset` функція шукає тільки **абсолютні дублікати** — рядки, де збігаються значення в абсолютно всіх стовпцях.
* З параметром `subset` перевірка виконується лише за вказаним переліком стовпців (наприклад, унікальним ID замовлення). При цьому значення в інших стовпцях можуть відрізнятися.


3. **Чому автоматичне видалення значень поза межами IQR не завжди правильне?**
* Правило IQR є лише математичною формальністю для виявлення кандидатів у викиди. Значення за межами межі може бути як помилкою даних, так і **реальним рідкісним фактом** (наприклад, оптова закупівля чи святковий сплеск продажів). Автоматичне вилучення таких даних без аналізу предметної області призводить до втрати цінної інформації про реальні бізнес-процеси.